# Simulación Monte Carlo de un portafolio de inversión
**Ana Paula Moreno Haro**

- Tenemos 100,000 pesos para invertir un año en NVIDIA, BIMBO, CEMEX y AstraZeneca.
- Simulamos cuánto podría valer el portafolio y comparamos tres formas de repartir el dinero.

## Datos
| i | Activo | Precio inicial (S0) |
|---|--------|---------------------|
| 0 | NVDA | 202.06 USD |
| 1 | BIMBOA | 66.18 MXN |
| 2 | CEMEXCPO | 20.28 MXN |
| 3 | AZN | 186.58 USD |
| x | USD/MXN | 18.58 MXN |

- Inversión: V0 = 100,000 MXN
- Horizonte: T = 1 año
- Simulaciones: M = 10,000

## Escenarios
- **A:** todo en NVIDIA → w = [1, 0, 0, 0]
- **B:** partes iguales → w = [0.25, 0.25, 0.25, 0.25]
- **C:** más peso en cada acción siguiente → w = [0.1, 0.2, 0.3, 0.4]

## Pasos
1. Bajamos 3 años de precios de las acciones y del dólar.
2. Sacamos μ (lo que suele ganar) y σ (qué tanto se mueve) con rendimientos logarítmicos.
3. Simulamos 10,000 precios posibles a un año.
4. Calculamos cuántas acciones compramos: Nᵢ = wᵢ · V0 / S0ᵢ
5. Calculamos el valor final: V_T = Σ Nᵢ · S_Tᵢ

- NVDA y AZN están en dólares, así que las pasamos a pesos con el dólar de hoy (18.58) y con el dólar simulado a un año.
- Fijamos las fechas de descarga y la semilla para que los resultados no cambien cada vez que corremos.

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

tickers = ['NVDA', 'BIMBOA.MX', 'CEMEXCPO.MX', 'AZN', 'USDMXN=X']
S0 = np.array([202.06, 66.18, 20.28, 186.58, 18.58])
V0 = 100000
M = 10000
T = 1

# fechas fijas para que los datos no cambien
inicio = '2023-11-11'
fin = '2026-11-12'

In [ ]:
datos = yf.download(tickers, start=inicio, end=fin)['Close']
datos = datos[tickers]
datos = datos.dropna()

- Sacamos cuánto sube o baja cada activo por día (rendimientos logarítmicos).
- El promedio es μ y la desviación es σ, las dos anualizadas.

In [ ]:
rend = np.log(datos / datos.shift(1)).dropna()

mu = rend.mean().values * 252
sigma = rend.std().values * np.sqrt(252)

print(pd.DataFrame({'mu': mu, 'sigma': sigma}, index=tickers))

- Simulamos 10,000 precios posibles a un año para cada activo, cada uno por separado (independientes).

In [ ]:
np.random.seed(0)
ST = np.zeros((M, 5))

for i in range(5):
    Z = np.random.normal(0, 1, M)
    ST[:, i] = S0[i] * np.exp(mu[i] * T + sigma[i] * np.sqrt(T) * Z)

- Pasamos NVDA y AZN a pesos: hoy con 18.58 y a un año con el dólar simulado.
- BIMBO y CEMEX ya están en pesos.

In [ ]:
tc0 = S0[4]
tcT = ST[:, 4]

S0_mxn = np.array([S0[0] * tc0, S0[1], S0[2], S0[3] * tc0])

ST_mxn = np.zeros((M, 4))
ST_mxn[:, 0] = ST[:, 0] * tcT
ST_mxn[:, 1] = ST[:, 1]
ST_mxn[:, 2] = ST[:, 2]
ST_mxn[:, 3] = ST[:, 3] * tcT

- Para cada escenario calculamos cuántas acciones compramos (N) y cuánto valen en un año (VT).
- Sacamos promedio, desviación, peor y mejor caso y probabilidad de perder.

In [ ]:
escenarios = {
    'A': np.array([1, 0, 0, 0]),
    'B': np.array([0.25, 0.25, 0.25, 0.25]),
    'C': np.array([0.1, 0.2, 0.3, 0.4])
}

resultados = {}
VT_todos = {}

for nombre, w in escenarios.items():
    N = w * V0 / S0_mxn
    VT = np.sum(N * ST_mxn, axis=1)
    VT_todos[nombre] = VT

    resultados[nombre] = {
        'VT promedio': VT.mean(),
        'Desv. est.': VT.std(),
        'Percentil 5%': np.percentile(VT, 5),
        'Percentil 95%': np.percentile(VT, 95),
        'Prob. perder': np.mean(VT < V0)
    }

tabla = pd.DataFrame(resultados).T
print(tabla)

- Cuántas acciones compramos de cada empresa en cada escenario.

In [ ]:
acciones = {}

for nombre, w in escenarios.items():
    N = w * V0 / S0_mxn
    acciones[nombre] = N

tabla_acciones = pd.DataFrame(acciones, index=['NVDA', 'BIMBOA', 'CEMEXCPO', 'AZN']).T
print(tabla_acciones.round(2))

- Si solo compramos acciones completas, redondeamos hacia abajo y vemos cuánto dinero nos sobra.

In [ ]:
for nombre, w in escenarios.items():
    N = w * V0 / S0_mxn
    N_enteras = np.floor(N)
    gastado = np.sum(N_enteras * S0_mxn)
    sobra = V0 - gastado

    print('Escenario', nombre)
    print('Acciones enteras:', N_enteras)
    print('Dinero gastado:', round(gastado, 2))
    print('Dinero que sobra:', round(sobra, 2))
    print()

In [ ]:
# cuál escenario deja más dinero en promedio
mejor = tabla['VT promedio'].idxmax()
print('Escenario que maximiza VT promedio:', mejor)

- Histograma de cada escenario. La línea punteada son los 100,000; a la izquierda perdemos.

In [ ]:
for nombre in VT_todos:
    plt.hist(VT_todos[nombre], bins=100, alpha=0.5, label='Escenario ' + nombre)

plt.axvline(V0, color='black', linestyle='--', label='V0')
plt.xlabel('VT (MXN)')
plt.ylabel('Frecuencia')
plt.title('Distribución del valor del portafolio en T = 1')
plt.legend()
plt.show()

## Conclusión
- A (todo en NVDA) es el que más gana en promedio, pero también el más arriesgado: tiene la mayor probabilidad de perder (18%) porque si NVDA cae no hay nada que compense.
- B reparte el dinero entre acciones de México y EU, tiene menos variación, la menor probabilidad de perder (10.5%) y un peor caso menos malo que A.
- C también está diversificado, pero carga más peso en unas acciones que en otras y gana menos que B.
- B es el que mejor equilibra lo que podemos ganar con lo que arriesgamos.

# Clase 4 nov: Cholesky
- Usamos AAPL, MSFT y NVDA. NVDA ya la teníamos, así que solo bajamos AAPL y MSFT.
- Sacamos sus rendimientos y vemos qué tanto se mueven juntas con la matriz de correlación R.

In [ ]:
nuevos = yf.download(['AAPL', 'MSFT'], start=inicio, end=fin)['Close']

datos3 = pd.concat([nuevos['AAPL'], nuevos['MSFT'], datos['NVDA']], axis=1)
datos3 = datos3.dropna()

rend3 = np.log(datos3 / datos3.shift(1)).dropna()

R = rend3.corr()
print(R.round(4))

Pasos del pizarrón:
1. Sacamos Σ, la matriz de covarianza.
2. Cholesky: L · Lᵀ = Σ.
3. Simulamos epsilon: normales independientes de 3 × M.
4. Z = L · epsilon ya viene correlacionado.
- La transponemos para que cada fila sea una simulación y cada columna una acción.

In [ ]:
Sigma = rend3.cov().values
L = np.linalg.cholesky(Sigma)

mu3 = np.array([rend3['AAPL'].mean(), rend3['MSFT'].mean(), rend3['NVDA'].mean()]) * 252

np.random.seed(0)
epsilon = np.random.normal(0, 1, [3, M])

Z = 0 + L @ epsilon
ZT = Z.T

print(pd.DataFrame(ZT[:5], columns=['AAPL', 'MSFT', 'NVDA']).round(4))

- Con cada simulación calculamos el precio a un año: S_T = S0 · exp((μ − σ²/2)·T + σ·√T·Z)
- S0 es el último precio descargado y Z son los choques correlacionados.

In [ ]:
S0_3 = datos3.iloc[-1].values
sigma3 = rend3.std().values * np.sqrt(252)

ST3 = S0_3 * np.exp((mu3 - sigma3**2 / 2) * T + sigma3 * np.sqrt(T) * ZT)

print('Precio inicial:', S0_3.round(2))
print('Precio simulado en T=1:', ST3.mean(axis=0).round(2))
print(pd.DataFrame(ST3[:5], columns=['AAPL', 'MSFT', 'NVDA']).round(2))

## Conclusión
- AAPL, MSFT y NVDA son del mismo sector, así que suelen subir y bajar juntas.
- Cholesky convierte números aleatorios independientes en choques que se mueven juntos, como en la vida real.
- Así la simulación no subestima el riesgo.

# Tarea: independiente vs correlacionado
- En clase supusimos que NVDA, BIMBO, CEMEX y AZN eran independientes porque son de sectores distintos.
- Aquí revisamos si eso es cierto y qué cambia si sí metemos la correlación.

## 1. Ejercicio A con correlación
- Sacamos la matriz de correlación de los 5 factores con los mismos rendimientos de clase.
- Metemos el dólar porque NVDA y AZN las pasamos a pesos con él.

In [ ]:
pd.set_option('display.width', 200)   # para que las tablas salgan completas

R5 = rend.corr()
print(R5.round(4))

**¿Qué observamos?**
- Ninguna correlación es cero, así que la independencia solo es una aproximación.
- Todas son bajas (la más alta es 0.21) porque son de sectores distintos.
- Las más altas son NVDA–CEMEX (0.21) y BIMBO–CEMEX (0.19); CEMEX es la que más se mueve con las demás.
- Con el dólar todas salen casi en cero, así que el tipo de cambio se mueve prácticamente por su cuenta.

- Hacemos Cholesky con R para que L · Lᵀ = R.
- Usamos los mismos epsilon en los dos casos, así lo único que cambia es la correlación.
- Independiente: Z = epsilon. Correlacionado: Z = L · epsilon.
- Usamos R y no Σ porque σ ya va en la fórmula del precio.
- Como usamos la misma semilla, el caso independiente da igual que en clase.

In [ ]:
L5 = np.linalg.cholesky(R5.values)

np.random.seed(0)
eps = np.random.normal(0, 1, [5, M])

Z_ind = eps.T            # independientes
Z_cor = (L5 @ eps).T     # correlacionados

# checamos que los choques sí salgan correlacionados
print(pd.DataFrame(np.corrcoef(Z_cor.T), index=tickers, columns=tickers).round(2))

ST_ind = S0 * np.exp(mu * T + sigma * np.sqrt(T) * Z_ind)
ST_cor = S0 * np.exp(mu * T + sigma * np.sqrt(T) * Z_cor)

- La correlación de los choques simulados sale casi igual a la de los datos, entonces Cholesky sí funcionó.
- Pasamos NVDA y AZN a pesos con el dólar simulado, igual que en clase.

In [ ]:
def a_pesos(ST):
    tcT = ST[:, 4]
    ST_mxn = np.zeros((M, 4))
    ST_mxn[:, 0] = ST[:, 0] * tcT
    ST_mxn[:, 1] = ST[:, 1]
    ST_mxn[:, 2] = ST[:, 2]
    ST_mxn[:, 3] = ST[:, 3] * tcT
    return ST_mxn

ST_ind_mxn = a_pesos(ST_ind)
ST_cor_mxn = a_pesos(ST_cor)

- Compramos las mismas acciones que en clase y calculamos VT en los dos casos.
- Sacamos promedio, desviación, percentiles 5% y 95% y probabilidad de perder.

In [ ]:
def resumen(VT):
    return {
        'VT promedio': VT.mean(),
        'Desv. est.': VT.std(),
        'Percentil 5%': np.percentile(VT, 5),
        'Percentil 95%': np.percentile(VT, 95),
        'Prob. perder': np.mean(VT < V0)
    }

VT_ind = {}
VT_cor = {}
resultados1 = {}

for nombre, w in escenarios.items():
    N = w * V0 / S0_mxn
    VT_ind[nombre] = np.sum(N * ST_ind_mxn, axis=1)
    VT_cor[nombre] = np.sum(N * ST_cor_mxn, axis=1)

    resultados1[nombre + ' indep'] = resumen(VT_ind[nombre])
    resultados1[nombre + ' corr'] = resumen(VT_cor[nombre])

tabla1 = pd.DataFrame(resultados1).T
print(tabla1.round(4))
print()

for nombre in escenarios:
    cambio = (VT_cor[nombre].std() / VT_ind[nombre].std() - 1) * 100
    print('Escenario', nombre, '-> cambio en la desv. est. al meter correlación:', round(cambio, 2), '%')

- Comparamos la distribución de cada escenario: independiente vs correlacionado.
- La línea punteada son los 100,000; a la izquierda perdemos.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))

for i, nombre in enumerate(escenarios):
    ax[i].hist(VT_ind[nombre], bins=100, alpha=0.5, label='Independiente')
    ax[i].hist(VT_cor[nombre], bins=100, alpha=0.5, label='Correlacionado')
    ax[i].axvline(V0, color='black', linestyle='--', label='V0')
    ax[i].set_title('Escenario ' + nombre + '  w = ' + str(escenarios[nombre]))
    ax[i].set_xlabel('VT (MXN)')
    ax[i].legend()

ax[0].set_ylabel('Frecuencia')
plt.tight_layout()
plt.show()

**¿Qué observamos?**
- El promedio casi no cambia (B: 131,609 vs 131,648), porque la correlación no cambia lo que gana cada acción, solo si se mueven juntas.
- En A (todo en NVDA) casi no hay diferencia (+0.7% en la desviación), porque solo hay una acción y su correlación con el dólar es casi cero.
- En B y C la distribución se abre: la desviación sube 11% en B y 13% en C.
- El peor caso empeora: el percentil 5% de B baja de 93,513 a 89,171.
- La probabilidad de perder sube de 10.5% a 13.7% en B y de 12.1% a 15.7% en C.
- Aunque las correlaciones son bajas, suponer independencia sí subestima el riesgo. Aun así, B sigue siendo la mejor opción.

## 2. AAPL, MSFT y NVDA con 100,000 MXN
- Usamos los datos de la clase del 4 de nov y agregamos el dólar, porque las tres están en dólares.
- Sacamos μ, σ, el último precio y la matriz de correlación.

In [ ]:
datos4 = pd.concat([datos3, datos['USDMXN=X']], axis=1)
datos4 = datos4.dropna()

rend4 = np.log(datos4 / datos4.shift(1)).dropna()

mu4 = rend4.mean().values * 252
sigma4 = rend4.std().values * np.sqrt(252)
S0_4 = datos4.iloc[-1].values
R4 = rend4.corr()

print(pd.DataFrame({'S0': S0_4, 'mu': mu4, 'sigma': sigma4}, index=datos4.columns).round(4))
print()
print(R4.round(4))

**¿Qué observamos?**
- Las correlaciones entre AAPL, MSFT y NVDA van de 0.26 a 0.42, más altas que en la parte 1, porque las tres son tecnológicas.
- Con el dólar otra vez salen casi en cero.
- Aquí la independencia ya no es una buena suposición.

- Simulamos igual que en la parte 1: mismos epsilon, y para el correlacionado Z = L · epsilon.
- Pasamos las tres acciones a pesos con el dólar simulado.

In [ ]:
L4 = np.linalg.cholesky(R4.values)

np.random.seed(0)
eps4 = np.random.normal(0, 1, [4, M])

Z4_ind = eps4.T
Z4_cor = (L4 @ eps4).T

ST4_ind = S0_4 * np.exp(mu4 * T + sigma4 * np.sqrt(T) * Z4_ind)
ST4_cor = S0_4 * np.exp(mu4 * T + sigma4 * np.sqrt(T) * Z4_cor)

# a pesos: precio en dólares por el tipo de cambio (columna 3)
tc0_4 = S0_4[3]
S0_4_mxn = S0_4[:3] * tc0_4

ST4_ind_mxn = np.zeros((M, 3))
ST4_cor_mxn = np.zeros((M, 3))
for i in range(3):
    ST4_ind_mxn[:, i] = ST4_ind[:, i] * ST4_ind[:, 3]
    ST4_cor_mxn[:, i] = ST4_cor[:, i] * ST4_cor[:, 3]

Esquemas de inversión:
- Caso 1: todo en AAPL → [1, 0, 0]
- Caso 2: casi partes iguales → [0.33, 0.33, 0.34]
- Caso 3: más peso en NVDA → [0.2, 0.2, 0.6]

In [ ]:
escenarios2 = {
    'Caso 1': np.array([1, 0, 0]),
    'Caso 2': np.array([0.33, 0.33, 0.34]),
    'Caso 3': np.array([0.2, 0.2, 0.6])
}

VT4_ind = {}
VT4_cor = {}
resultados2 = {}

for nombre, w in escenarios2.items():
    N = w * V0 / S0_4_mxn
    VT4_ind[nombre] = np.sum(N * ST4_ind_mxn, axis=1)
    VT4_cor[nombre] = np.sum(N * ST4_cor_mxn, axis=1)

    resultados2[nombre + ' indep'] = resumen(VT4_ind[nombre])
    resultados2[nombre + ' corr'] = resumen(VT4_cor[nombre])

    print(nombre, '-> acciones compradas:', N.round(2))

tabla2 = pd.DataFrame(resultados2).T
print()
print(tabla2.round(4))
print()

for nombre in escenarios2:
    cambio = (VT4_cor[nombre].std() / VT4_ind[nombre].std() - 1) * 100
    print(nombre, '-> cambio en la desv. est. al meter correlación:', round(cambio, 2), '%')

- Mismas gráficas que en la parte 1: independiente vs correlacionado.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))

for i, nombre in enumerate(escenarios2):
    ax[i].hist(VT4_ind[nombre], bins=100, alpha=0.5, label='Independiente')
    ax[i].hist(VT4_cor[nombre], bins=100, alpha=0.5, label='Correlacionado')
    ax[i].axvline(V0, color='black', linestyle='--', label='V0')
    ax[i].set_title(nombre + '  w = ' + str(escenarios2[nombre]))
    ax[i].set_xlabel('VT (MXN)')
    ax[i].legend()

ax[0].set_ylabel('Frecuencia')
plt.tight_layout()
plt.show()

**¿Qué observamos?**
- El promedio otra vez casi no cambia.
- Caso 1 (todo en AAPL): casi igual (+1.2% en la desviación), porque es una sola acción.
- Caso 2 (partes iguales): es donde más se nota. La desviación sube 18%, el percentil 5% baja de 93,166 a 85,406 y la probabilidad de perder sube de 8.9% a 14.3%.
- Caso 3 (60% NVDA): es el más disperso de los tres, y con correlación su desviación sube 8.7%. Se nota menos que en el caso 2 porque está más concentrado en una sola acción.
- Suponer independencia nos hace creer que el portafolio diversificado es más seguro de lo que realmente es.

# Conclusión
- La correlación no cambia cuánto ganamos en promedio, cambia el riesgo.
- Con una sola acción casi no importa, porque no hay otras con las que se mueva junto.
- En la parte 1, aunque las acciones son de sectores distintos y las correlaciones son bajas, el riesgo de los portafolios diversificados sube 11–13% al tomarlas en cuenta.
- En la parte 2 las tres son tecnológicas, las correlaciones son más altas y el riesgo del portafolio repartido sube casi 18%.
- Mientras más diversificado está el portafolio, más importa la correlación: la diversificación protege menos de lo que creemos si suponemos independencia.
- Conviene revisar siempre la matriz de correlación antes de suponer que los activos son independientes.